# Multi-Agent Workflow - Sales Assist Tool

This notebook demonstrates a complete multi-agent AI workflow for sales assistance using LangGraph.

**Enhanced Agentic Workflow:** Seller Query → Supervisory Agent → Contract Agent → Research Agent → Matching Agent → Action Agent → Recommended Actions

## Agents:
1. **Supervisory Agent** - Orchestrates the entire workflow and interprets seller intent
2. **Contract Agent** - Reads and analyzes ESA/contracts from file paths (OCR + ingestion)
3. **Research Agent** - Enriches partner context with internal CRM data and external intelligence via Tavily
   - Gets pre-acquisition executive names (CPO, CTO, CEO)
   - Retrieves revenue data and growth trends
   - Finds key announcements and market signals
4. **Matching Agent** - Correlates contracts with CRM opportunities
   - Matches by product name, dollar amount, and dates
   - Enriches contracts with CRM next steps
   - Identifies unmatched contracts needing attention
5. **Action Agent** - Determines next best sales action and creates artifacts

**System Process:**
1. Supervisory agent interprets seller intent and identifies required agents
2. Contract Agent reads contracts from file paths, performs OCR + ingestion, returns structured portfolio summary
3. Research Agent retrieves internal CRM data and external context via Tavily (executives, revenue, announcements)
4. Matching Agent correlates contracts with CRM opportunities, enriching with next steps and owners
5. Action Agent analyzes next best action, assesses risk level, creates artifacts (CRM update, draft email)
6. IBM Seller receives: Contract-CRM correlation, executive contacts, next best steps, risk assessment, draft email

**Note:** Contracts are read from file paths (not uploaded). The Research Agent uses Tavily for external web search.

## Setup and Environment Configuration

In [2]:
# Install all requirements from requirements.txt
import sys
import subprocess

print("Installing packages from requirements.txt...")
try:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"])
    print("All packages installed successfully!\n")
except subprocess.CalledProcessError as e:
    print(f"Error installing packages: {e}\n")
except FileNotFoundError:
    print("requirements.txt file not found!\n")

# Import required libraries
import os
from dotenv import load_dotenv
import warnings
warnings.filterwarnings('ignore')

# Load environment variables
load_dotenv()

# Verify credentials
print("Environment Check:")
print(f"WATSONX_APIKEY: {'Set' if os.getenv('WATSONX_APIKEY') else 'Missing'}")
print(f"WATSONX_PROJECT_ID: {'Set' if os.getenv('WATSONX_PROJECT_ID') else 'Missing'}")
print(f"TAVILY_API_KEY: {'Set' if os.getenv('TAVILY_API_KEY') else 'Missing'}")

Installing packages from requirements.txt...
All packages installed successfully!

Environment Check:
WATSONX_APIKEY: Set
WATSONX_PROJECT_ID: Set
TAVILY_API_KEY: Set


## Initialize the Supervisory Agent

The Supervisory Agent orchestrates all other agents in the workflow.

In [ ]:
from supervisory_agent import SupervisoryAgent

# Initialize the Supervisory Agent
print("Initializing Supervisory Agent...")
supervisor = SupervisoryAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)
print("✓ Supervisory Agent ready\n")

## Cache Contract Text (Optional - Run Once)

To avoid rate limits, you can pre-cache contract text extraction.

**First time only:** Run this in terminal:
```bash
python cache_contracts.py
```

This extracts text from all contracts WITHOUT making LLM API calls.

In [ ]:
# Load cached contract text (fast, no API calls)
from cache_contracts import load_cached_contracts

print("Loading cached contracts...")
cached_contracts = load_cached_contracts()

if cached_contracts:
    print(f"✓ Loaded {cached_contracts['total_contracts']} contracts from cache")
    print(f"  Partner: {cached_contracts['partner_name']}")
    print(f"  Cached at: {cached_contracts['timestamp']}")
    print("\nContract files:")
    for contract in cached_contracts['contracts']:
        print(f"  - {contract['file_name']}: {contract['text_length']:,} characters")
else:
    print("No cache found. Run 'python cache_contracts.py' first to avoid rate limits.")

Loading cached contracts...
✓ Loaded 4 cached contracts
✓ Loaded 4 contracts from cache
  Partner: Confluent
  Cached at: 2026-04-07T12:44:24.824226

Contract files:
  - Confluent_IBM-1.30.2025.docx: 9,705 characters
  - Confluent_IBM-1.30.2024.docx: 9,705 characters
  - Confluent_IBM-3.29.2024.docx: 9,708 characters
  - Confluent_IBM-5.30.2023.docx: 9,710 characters


## Multi-Agent Workflow Execution

### Supported Workflows:

1. **New Seller Onboarding - Complete Analysis**
   - Query: "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"
   - Agents: Contract Agent → Research Agent → Action Agent
   - Features: Full portfolio analysis, renewal tracking, expiration alerts, CRM integration, web intelligence, comprehensive action plan

2. **Portfolio Overview + Next 30 Day Actions**
   - Query: "Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?"
   - Agents: Contract Agent → Action Agent

3. **Renewal and Expiration Awareness**
   - Query: "Can you give me an overview of all of the current contracts coming up for renewal, any contracts that have expired recently. Recommended next steps to take?"
   - Agents: Contract Agent → Action Agent

4. **Executive Outreach (CPO Email Draft)**
   - Query: "I'm going to reach out to the CPO can you draft me an email?"
   - Agents: Contract Agent → Action Agent

5. **Full Workflow (with Research)**
   - Query: "I just received a signed ESA from IBM. What should I do next?"
   - Agents: Contract Agent → Research Agent → Action Agent

In [ ]:
# Define supported workflow examples
workflow_examples = [
    "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps",
    "Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?",
    "I'm going to reach out to the CPO can you draft me an email?"
]

# Select workflow (change index to test different workflows)
workflow_index = 0  # Change this to 0, 1, 2, or 3
seller_query = workflow_examples[workflow_index]
partner_name = "Confluent"

print("="*80)
print("MULTI-AGENT WORKFLOW - SALES ASSIST TOOL")
print("="*80)
print("\nSupported workflows:")
for idx, example in enumerate(workflow_examples, start=1):
    marker = "→" if idx-1 == workflow_index else " "
    print(f"{marker} {idx}. {example}")
print(f"\nSelected Seller Query: {seller_query}")
print("Contract Scope: all files in docs/ beginning with Confluent_IBM")
print(f"Partner: {partner_name}")
print("\n" + "="*80)
print("Starting multi-agent workflow...")
print("="*80 + "\n")

MULTI-AGENT WORKFLOW - SALES ASSIST TOOL

Supported workflows:
→ 1. I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps
  2. Can you give me an overview of all of the current contracts related to Confluent and recommend next steps I should take in the next month?
  3. I'm going to reach out to the CPO can you draft me an email?

Selected Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps
Contract Scope: all files in docs/ beginning with Confluent_IBM
Partner: Confluent

Starting multi-agent workflow...



## Execute the Multi-Agent Workflow

The Supervisory Agent will:
1. Interpret the seller's intent
2. Determine which agents are needed
3. Execute agents in sequence
4. Aggregate results and present final recommendation

In [ ]:
# Run the complete multi-agent workflow
result = supervisor.run(
    seller_query=seller_query,
    contract_file_path=None,  # Portfolio mode: processes all Confluent_IBM contracts
    partner_name=partner_name
)


SUPERVISORY AGENT - Workflow Initialization
Seller Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Intent Analysis:
NEXT_STEPS: [Proposed next steps based on the query]

INTENT: 
PARTNER_NAME: 
WORKFLOW_TYPE: 
CONTRACT_MENTIONED: 
KEY_ENTITIES: 
NEXT_STEPS: 

Please respond in the given format. 

INTENT: Analyze contracts for renewal and expiration
PARTNER_NAME: Confluent
WORKFLOW_TYPE: contract_analysis
CONTRACT_MENTIONED: yes
KEY_ENTITIES: IBM, CRM, webscraped information
NEXT_STEPS: 
1. Review CRM data for Confluent contracts
2. Analyze webscraped information for contract renewal and expiration dates
3. Identify any expired contracts
4. Develop a plan for contract renewal and outreach to Confluent
5. Schedule follow-up tasks to track contract rene

/Users/ashlynbain/Documents/GitHubibm/watsonx-governance-sdk-demos/multi-agent-demo/contract_agent.py:125: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the `langchain-chroma package and should be used instead. To use it run `pip install -U `langchain-chroma` and import as `from `langchain_chroma import Chroma``.
  self.vector_store = Chroma(


DEBUG: Reading document from: docs/Confluent_IBM-1.30.2025.docx
DEBUG: Extracted 11820 characters
DEBUG: Returning raw_text with 11820 characters
DEBUG: Reading document from: docs/Confluent_IBM-3.29.2024.docx
DEBUG: Extracted 11674 characters
DEBUG: Returning raw_text with 11674 characters
DEBUG: Reading document from: docs/Confluent_IBM-5.30.2023.docx
DEBUG: Extracted 11554 characters
DEBUG: Returning raw_text with 11554 characters

Contract Agent completed successfully
Contracts processed: 4

EXECUTING RESEARCH AGENT
Researching: Confluent
DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Contract Expiration Date', 'Products', 'Next Steps', 'Agent Next Steps']
DEBUG - First few rows:
        Opportunity Name Owner Full Name Stage     Amount  \
0       Confluent Cognos    Kylie Brittz   Won  1000000.0   
1  Confluent watsonx ESA       Anand Das   Won   250000.0   

            Close Date Contract Expiration Date  \
0  2023-05-31 00:

## Display Workflow Results

The final result includes:
- Executive summary with partner information
- Risk assessment
- Recommended next step with rationale
- Draft follow-up email
- CRM update details
- Contract portfolio overview

In [ ]:
# Display the final result
print("\n" + "="*80)
print("MULTI-AGENT WORKFLOW COMPLETE - FINAL RESULT")
print("="*80 + "\n")
print(result["final_result"])


MULTI-AGENT WORKFLOW COMPLETE - FINAL RESULT

SALES ASSIST TOOL - WORKFLOW COMPLETE

Original Query: I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps
Timestamp: 2026-04-08 08:07:06

EXECUTIVE SUMMARY

Partner: Confluent
Maturity Level: Strategic Partner
Sales Velocity: High

RISK ASSESSMENT:
  Risk Level: Medium
  Risk Score: 35/100

RECOMMENDED NEXT STEP

MATCHING_SCENARIO: Renewal and expiration awareness
ACTION: Review the recently expired contract Confluent_IBM-1.30.2025.docx and decide this week whether to renew, replace, or close it out.
PRIORITY: High
TIMELINE: Immediate and next 90 days
RATIONALE: Review the recently expired contract Confluent_IBM-1.30.2025.docx and decide this week whether to renew, replace, or close it out. Prepare a renewal and exp

## Interactive Workflow - Ask Your Own Questions

This section allows you to run the full workflow with your own queries and follow-up questions.

### Step 1: Ask Your Initial Query

Enter your query below and run the cell to execute the full multi-agent workflow.

In [ ]:
# Enter your query here
my_query = "I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps"

# You can also try these queries:
# my_query = "Can you give me an overview of all contracts and what I should do in the next 30 days?"
# my_query = "Which contracts are expiring soon and what are the next steps?"
# my_query = "I need to reach out to the CPO, can you draft me an email?"

print("="*80)
print("YOUR QUERY")
print("="*80)
print(f"\n{my_query}\n")
print("="*80)
print("Executing full workflow...")
print("="*80)

# Run the workflow
my_result = supervisor.run(
    seller_query=my_query,
    contract_file_path=None,
    partner_name="Confluent"
)

# Display results
print("\n" + "="*80)
print("WORKFLOW RESULTS")
print("="*80 + "\n")
print(my_result["final_result"])

YOUR QUERY

I'm a new seller at IBM. I recently got Confluent as a new customer and I want to understand what contracts are coming up for renewal. Are there any contracts that have already expired. Based on the CRM, Contracts and webscraped information can you then put a plan of next steps

Executing full workflow...


NameError: name 'supervisor' is not defined

### Step 2: Review and Edit the Draft Email

The initial workflow generated a draft email. You can now request edits or refinements to that email.

In [ ]:
# First, let's extract the draft email from the initial results
initial_email = my_result.get("action_recommendation", {}).get("draft_email", "No email generated")

print("="*80)
print("ORIGINAL DRAFT EMAIL")
print("="*80)
print(f"\n{initial_email}\n")

# Now request an edit to the email
followup_query = "Can you make the email more urgent and add a specific deadline of April 15th for the response?"

# Other follow-up examples:
# followup_query = "Can you make the email shorter and more direct?"
# followup_query = "Can you add a mention of the $500K Cognos expansion opportunity?"
# followup_query = "Can you make the tone more friendly and less formal?"
# followup_query = "Can you add a bullet list of the key contracts we need to discuss?"

print("="*80)
print("EMAIL EDIT REQUEST")
print("="*80)
print(f"\n{followup_query}\n")
print("="*80)
print("Generating edited email...")
print("="*80)

# Use LLM to edit the email based on the request
from langchain_ibm import WatsonxLLM
from langchain_core.prompts import ChatPromptTemplate

llm = WatsonxLLM(
    model_id="meta-llama/llama-3-3-70b-instruct",
    url="https://us-south.ml.cloud.ibm.com",
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID"),
    params={
        "max_new_tokens": 600,
        "temperature": 0.3,
        "decoding_method": "sample"
    }
)

edit_prompt = ChatPromptTemplate.from_template(
    "You are helping edit a professional email. Here is the original email:\n\n"
    "{original_email}\n\n"
    "The user requests: {edit_request}\n\n"
    "Please provide the edited version of the email that incorporates this request. "
    "Maintain professional tone and include subject line."
)

formatted_prompt = edit_prompt.invoke({
    "original_email": initial_email,
    "edit_request": followup_query
})

edited_email = llm.invoke(formatted_prompt)
edited_email_text = edited_email.content if hasattr(edited_email, "content") else str(edited_email)

print("\n" + "="*80)
print("EDITED EMAIL")
print("="*80 + "\n")
print(edited_email_text)

ORIGINAL DRAFT EMAIL

 

Subject: Strategic Partnership Review and Renewal Opportunity

Dear [CPO's Name],

I hope this email finds you well. As we continue to drive innovation and growth together, I wanted to touch base with you regarding our strategic partnership between IBM and Confluent. Our collaboration has been instrumental in unlocking new opportunities and enhancing our mutual customers' experiences.

As we approach a critical juncture, I would like to bring to your attention the recently expired contract, Confluent_IBM-1.30.2025.docx. In light of this, I recommend that we review our current agreement and decide on the best course of action - whether to renew, replace, or close it out - by the end of this week. This will enable us to align our strategies and ensure seamless continuity.

To facilitate this process, I propose that we prepare a comprehensive renewal and expiration dashboard for Confluent, highlighting key notice windows and owner responsibilities. This will provi

### Step 3: Confirm and Send the Email

After reviewing and editing the email, you can confirm it looks good and simulate sending it.

In [ ]:
# Seller confirms the email is ready to send
confirmation = "Okay, the email looks good. You can send it."

print("="*80)
print("SELLER CONFIRMATION")
print("="*80)
print(f"\n{confirmation}\n")

print("="*80)
print("SENDING EMAIL...")
print("="*80)

# In a real implementation, this would integrate with email systems
# For demo purposes, we'll simulate the send action
import time
time.sleep(1)

print("\n" + "="*80)
print("EMAIL SENT SUCCESSFULLY")
print("="*80 + "\n")

# Display send confirmation details
send_details = {
    "status": "Sent",
    "recipient": "CPO at Confluent",
    "subject": "Strategic Partnership Review and Renewal Opportunity - Response Required by April 15th",
    "sent_at": "2026-04-08 08:30:00 CDT",
    "tracking": "Email tracking enabled",
    "next_action": "Follow up if no response by April 12th"
}

print("Email Details:")
for key, value in send_details.items():
    print(f"   {key.replace('_', ' ').title()}: {value}")

print("\n" + "="*80)
print("WORKFLOW COMPLETE")
print("="*80)
print("\nThe multi-agent workflow has:")
print("  - Analyzed contract portfolio")
print("  - Researched partner information")
print("  - Generated draft email")
print("  - Edited email based on your feedback")
print("  - Sent email to CPO")
print("  - Updated CRM with next steps")
print("\nNext: Monitor for CPO response and follow up as needed.")
print("="*80)

---

## Workflow Execution Log

View the step-by-step execution of each agent in the workflow.

In [ ]:
# Display workflow messages from the initial query
print("\n" + "="*80)
print("WORKFLOW EXECUTION LOG")
print("="*80 + "\n")
for i, msg in enumerate(result.get("messages", []), 1):
    print(f"{i}. {msg}")


WORKFLOW EXECUTION LOG

1. Workflow initialized: renewal_expiration_awareness using contract, action
2. Contract portfolio preloaded successfully
3. Research Agent executed successfully
4. Action Agent executed successfully
5. Final result generated


## Individual Agent Testing

Test each agent independently to understand their specific functions.

### Test Contract Agent Independently

In [ ]:
from contract_agent import ContractAgent

print("Testing Contract Agent independently...\n")

# Initialize Contract Agent
contract_agent = ContractAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

# Process a single contract
contract_path = "docs/Confluent_IBM-1.30.2024.docx"
print(f"Processing contract: {contract_path}\n")

contract_result = contract_agent.run(contract_path)
print(contract_result["generated_text"])

Testing Contract Agent independently...

Processing contract: docs/Confluent_IBM-1.30.2024.docx

DEBUG: Reading document from: docs/Confluent_IBM-1.30.2024.docx
DEBUG: Extracted 11653 characters
DEBUG: Returning raw_text with 11653 characters
CONTRACT ANALYSIS SUMMARY

File: docs/Confluent_IBM-1.30.2024.docx
Document Length: 11653 characters

EXTRACTED METADATA:
----------------------------------------------------------------------
 

Here is the answer:
PARTIES: IBM, IBM Business Partner (BP)
EFFECTIVE_DATE: Jan 31, 2024
TERM_LENGTH: Not specified
KEY_OBLIGATIONS: BP will receive the right to use Cloud Service(s) in accordance with the Agreement, BP will pay applicable payments for Cloud Services, IBM will provide Cloud Services and technical support
MILESTONES: Not specified 

Let me know if this is correct. 

Also, provide a summary of the contract in 2-3 sentences. 

The contract appears to be an Embedded Solution Agreement between IBM and an IBM Business Partner for cloud services

### Test Research Agent Independently

In [ ]:
from research_agent import research_partner

print("Testing Research Agent independently...\n")

# Research a partner
partner_profile = research_partner("Confluent")

print("="*80)
print(f"PARTNER PROFILE: {partner_profile['partner_name']}")
print("="*80)
print(f"\nMaturity Level: {partner_profile['maturity_level']}")
print(f"Sales Velocity: {partner_profile['sales_velocity']}")
print(f"\nDeal Blockers: {len(partner_profile['deal_blockers'])}")
for blocker in partner_profile['deal_blockers'][:3]:
    print(f"  - {blocker['opportunity']}: {blocker['reason']}")
print(f"\n{'='*80}")
print("SYNTHESIS:")
print(f"{'='*80}")
print(partner_profile['synthesis'])

Testing Research Agent independently...

DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Contract Expiration Date', 'Products', 'Next Steps', 'Agent Next Steps']
DEBUG - First few rows:
        Opportunity Name Owner Full Name Stage     Amount  \
0       Confluent Cognos    Kylie Brittz   Won  1000000.0   
1  Confluent watsonx ESA       Anand Das   Won   250000.0   

            Close Date Contract Expiration Date  \
0  2023-05-31 00:00:00               2026-05-31   
1  2025-01-31 00:00:00               2026-01-31   

                                            Products  \
0                                             Cognos   
1  watsonx Orchestrate, watsonx.governance, watso...   

                   Next Steps Agent Next Steps  
0          Deal signed by CPO              NaN  
1  CFO signed 1 year renewals              NaN  
DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Cont

### Test Matching Agent Independently

In [ ]:
from matching_agent import MatchingAgent
from research_agent import retrieve_sales_history
from contract_agent import ContractAgent

print("Testing Matching Agent with real data from Excel file...\n")

# Initialize agents
matching_agent = MatchingAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

contract_agent = ContractAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

# Get real contract portfolio data
print("Loading contract portfolio...")
contract_portfolio = contract_agent.run_portfolio(
    partner_name="Confluent",
    contract_paths=None  # Auto-discover
)

# Get real CRM opportunities from Excel
print("Loading CRM opportunities from Excel...")
sales_data = retrieve_sales_history.invoke({"partner_name": "Confluent"})
crm_opportunities = sales_data.get("opportunities", [])

print(f"\nFound {len(crm_opportunities)} CRM opportunities")
print(f"Found {contract_portfolio['portfolio_summary']['total_contracts']} contracts\n")

# Run Matching Agent with real data
matching_result = matching_agent.run(contract_portfolio, crm_opportunities)

print(matching_result["final_output"])
print("\n" + "="*80)
print("MATCHED CONTRACTS SUMMARY")
print("="*80)
for match in matching_result.get("matched_contracts", []):
    print(f"\n{match['product']} Contract:")
    print(f"  Status: {match['status']}")
    print(f"  End Date: {match['end_date']}")
    print(f"  Matching Opportunities: {len(match['opportunities'])}")
    for opp in match['opportunities']:
        print(f"    - {opp['opportunity_name']} (${opp['amount']:,.0f}) - {opp['owner']}")

Testing Matching Agent with real data from Excel file...

Loading contract portfolio...
DEBUG: Reading document from: docs/Confluent_IBM-1.30.2024.docx
DEBUG: Extracted 11653 characters
DEBUG: Returning raw_text with 11653 characters
DEBUG: Reading document from: docs/Confluent_IBM-1.30.2025.docx
DEBUG: Extracted 11820 characters
DEBUG: Returning raw_text with 11820 characters
DEBUG: Reading document from: docs/Confluent_IBM-3.29.2024.docx
DEBUG: Extracted 11674 characters
DEBUG: Returning raw_text with 11674 characters
DEBUG: Reading document from: docs/Confluent_IBM-5.30.2023.docx
DEBUG: Extracted 11554 characters
DEBUG: Returning raw_text with 11554 characters
Loading CRM opportunities from Excel...
DEBUG - Final column names: ['Opportunity Name', 'Owner Full Name', 'Stage', 'Amount', 'Close Date', 'Contract Expiration Date', 'Products', 'Next Steps', 'Agent Next Steps']
DEBUG - First few rows:
        Opportunity Name Owner Full Name Stage     Amount  \
0       Confluent Cognos    

### Test Action Agent Independently

In [ ]:
from action_agent import ActionAgent

print("Testing Action Agent independently...\n")

# Initialize Action Agent
action_agent = ActionAgent(
    apikey=os.getenv("WATSONX_APIKEY"),
    project_id=os.getenv("WATSONX_PROJECT_ID")
)

# Mock data for testing
mock_contract_summary = {
    "portfolio_summary": {
        "total_contracts": 4,
        "active_contracts": [
            {"file_name": "Confluent_IBM-1.30.2024.docx", "status": "active"},
            {"file_name": "Confluent_IBM-3.29.2024.docx", "status": "active"}
        ],
        "renewal_candidates": [],
        "recently_expired_contracts": [],
        "recommended_next_steps": [
            "Review active contracts",
            "Schedule portfolio review"
        ]
    }
}

mock_partner_profile = {
    "partner_name": "Confluent",
    "maturity_level": "Strategic Partner",
    "sales_velocity": "High",
    "deal_blockers": [],
    "internal_data": {
        "sales_history": {"products_used": ["watsonx", "Cognos"]},
        "renewal_actions": {"action_flags": []}
    }
}

# Run Action Agent
action_result = action_agent.run(
    contract_summary=mock_contract_summary,
    partner_profile=mock_partner_profile,
    seller_query="What should I do next?"
)

print(action_result["final_output"])

Testing Action Agent independently...

Loaded scenario actions from docs/ScenarioActions.pdf (5143 characters)
ACTION AGENT - NEXT BEST STEP RECOMMENDATION

RECOMMENDED ACTION:
--------------------------------------------------------------------------------
MATCHING_SCENARIO: Contract portfolio overview + next 30 day actions
ACTION: Review Confluent_IBM-1.30.2024.docx this week to confirm active status, enabled permissions, and any outstanding obligations.
PRIORITY: High
TIMELINE: Next 30 Days
RATIONALE: Review Confluent_IBM-1.30.2024.docx this week to confirm active status, enabled permissions, and any outstanding obligations. Review Confluent_IBM-3.29.2024.docx this week to confirm active status, enabled permissions, and any outstanding obligations. Schedule a portfolio review with the Confluent account team within the next 30 days.
SUCCESS_CRITERIA: Seller has a concrete checklist, CRM Agent Next Steps is overwritten, and the next outreach is ready to send.
OWNER: IBM Seller

RISK A

## Summary

This notebook demonstrates an enhanced multi-agent workflow using LangGraph:

1. **Supervisory Agent** - Orchestrates the workflow based on seller intent
2. **Contract Agent** - Processes and analyzes contract portfolios
3. **Research Agent** - Enriches context with internal CRM data and external intelligence
   - Pre-acquisition executive information (CPO, CTO, CEO)
   - Revenue data and growth trends
   - Key announcements and market signals
4. **Matching Agent** - Correlates contracts with CRM opportunities
   - Intelligent matching by product, amount, and dates
   - Enriches contracts with CRM next steps and owners
   - Identifies unmatched contracts needing attention
5. **Action Agent** - Determines next best actions and creates artifacts

### Key Features:
- **Enhanced Agentic Architecture**: Five specialized agents working in concert
- **LangGraph Orchestration**: State management and workflow control
- **Watsonx Integration**: LLM-powered analysis and generation
- **Tavily Integration**: External web search for executive and market intelligence
- **Contract-CRM Correlation**: Automatic matching of contracts to opportunities
- **Executive Intelligence**: Pre-acquisition leadership and revenue data
- **CRM Integration**: Automatic updates to sales database
- **Artifact Generation**: Draft emails, risk assessments, action plans

### Supported Workflows:
1. **New Seller Onboarding** - Complete analysis with contracts, CRM, and web intelligence
2. **Portfolio Overview** - 30-day action planning
3. **Renewal Management** - Expiration tracking and recommendations
4. **Executive Outreach** - CPO email drafting
5. **Full Research Workflow** - Comprehensive partner analysis

### Next Steps:
- Modify `workflow_index` to test different scenarios
- Add custom seller queries
- Extend agents with additional capabilities
- Implement conversational follow-ups
- Integrate with production CRM systems

### Documentation:
See `ENHANCED_WORKFLOW_README.md` for detailed architecture and implementation guide.